<a href="https://colab.research.google.com/github/basilhakim/NLP-classworks/blob/main/NLPword2vec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ASSIGNMENT 2: Word2Vec Text Completion and Word Cloud Visualization (2215315)

Part A: Sentence Completion using Word2Vec:

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 59.5 MB/s eta 0:00:00


In [ ]:
import random
from gensim.models import Word2Vec

In [ ]:
# Prepare example sentences with a missing word placeholder
example_sentences = [
    "The sun rises in the ____.",
    "Birds fly in the ____.",
    "Fish swim in the ____.",
    "I like to read ____."
]

# Define the placeholder
placeholder = "____"

print("Example sentences prepared:")
for sentence in example_sentences:
  print(sentence)

Example sentences prepared:
The sun rises in the ____.
Birds fly in the ____.
Fish swim in the ____.
I like to read ____.


In [ ]:
import numpy as np

def predict_missing_word(sentence, model, placeholder):
  """
  Predicts the most likely missing word in a sentence using Word2Vec.

  Args:
    sentence (str): The sentence with a placeholder for the missing word.
    model: The trained Word2Vec model.
    placeholder (str): The placeholder string representing the missing word.

  Returns:
    str: The predicted missing word, or None if prediction is not possible.
  """
  words = sentence.replace('.', '').split()
  if placeholder not in words:
    return None # Placeholder not found

  placeholder_index = words.index(placeholder)
  context_words = [word for word in words if word != placeholder and word in model.key_to_index]

  if not context_words:
    return None # No context words found in the model vocabulary

  # Calculate the average vector of the context words
  context_vector = np.mean([model[word] for word in context_words], axis=0)

  # Find the most similar word to the context vector in the model's vocabulary
  # Exclude words already in the sentence
  words_in_sentence = set(words)

  try:
    # Find the most similar words to the context vector
    similar_words = model.most_similar(positive=[context_vector], topn=20) # Get top 20 similar words
    for word, similarity in similar_words:
        if word not in words_in_sentence:
            return word # Return the first most similar word not in the sentence
    return None # No suitable word found
  except KeyError:
    return None # Handle cases where the context vector might cause an error

In [ ]:
import gensim.downloader as api

# Load the pre-trained Word2Vec model (e.g., Google News vectors)
# This can take some time depending on the model size and your internet connection.
print("Loading Word2Vec model...")
word2vec_model = api.load("word2vec-google-news-300")
print("Model loaded successfully!")

Loading Word2Vec model...
[==================================================] 100.0% 1662.8/1662.8MB downloaded
Model loaded successfully!


In [ ]:
# Test the predict_missing_word function
print("\nTesting the prediction function:")
for sentence in example_sentences:
  predicted_word = predict_missing_word(sentence, word2vec_model, placeholder)
  if predicted_word:
    completed_sentence = sentence.replace(placeholder, predicted_word)
    print(f"Original: {sentence}")
    print(f"Predicted: {completed_sentence}\n")
  else:
    print(f"Could not predict for: {sentence}\n")


Testing the prediction function:
Original: The sun rises in the ____.
Predicted: The sun rises in the Thick_smoke_obscured.

Original: Birds fly in the ____.
Predicted: Birds fly in the flies.

Original: Fish swim in the ____.
Predicted: Fish swim in the swimming.

Original: I like to read ____.
Predicted: I like to read you.



# **Explanation of Word Embeddings**
Word embeddings are a technique used in natural language processing (NLP) to represent words as dense vectors in a continuous vector space. Unlike traditional methods like one-hot encoding where each word is represented by a sparse vector with a single '1' and the rest '0's, word embeddings capture semantic relationships between words.

Here's how they work and how they execute tasks like predicting missing words:

**Vector Representation:** Each word is mapped to a point (a vector) in a multi-dimensional space. The dimensions of this space represent different semantic and syntactic features.

**Semantic Relationships:** The key idea behind word embeddings is that words with similar meanings or that are used in similar contexts are located closer to each other in this vector space. The position and distance between vectors capture these relationships. For example, the vector for "king" might be close to the vector for "queen," and the vector for "Paris" might be close to the vector for "France."

**Vector Arithmetic:** Interestingly, vector arithmetic can capture analogies. For example, the vector operation vector("king") - vector("man") + vector("woman") often results in a vector that is very close to vector("queen").

**Predicting Missing Words:** In the context of sentence completion, word embeddings allow us to predict missing words based on the surrounding words (context). By averaging or combining the vectors of the known words in a sentence, we can create a context vector. We can then find the word in the vocabulary whose embedding vector is most similar to this context vector. Because semantically related words have similar vectors, the word whose vector is closest to the context vector is likely the most appropriate word to fill the blank. The predict_missing_word function uses this principle by calculating a context vector from the words around the placeholder and then finding the most similar word in the model's vocabulary. (Explanation by Gemini)